# PyTorch Fundamentals: BatchNorm variance: biased vs unbiased

**Solution notebook — Delta Drills #455**

Run the cells top-to-bottom to see the reference answer execute.


## Problem

Prove that BatchNorm2d in train mode uses the BIASED variance. Write solve(x) for a (B, C, H, W) float tensor: build `bn = torch.nn.BatchNorm2d(C, affine=False, eps=1e-5)` (C from x's channel dim), put it in train mode, and run `out = bn(x)`. Then normalize `x` yourself: per-channel mean and BIASED variance over the non-channel dims with `keepdim=True`, then (x − mean)/sqrt(var + eps). Return `(matches, out)` where `matches = bool(torch.allclose(out, manual, atol=1e-5))`.


<details><summary>💡 Hint (click to reveal)</summary>

BN with `affine=False`, `.train()`; manual: mean/var over (0,2,3) keepdim, biased, then (x−μ)/√(σ²+ε).

</details>


In [ ]:
%pip install -q numpy torch --index-url https://download.pytorch.org/whl/cpu

## Reference solution


In [ ]:
import torch
import torch.nn as nn

def solve(x):
    eps = 1e-5
    bn = nn.BatchNorm2d(x.shape[1], affine=False, eps=eps)
    bn.train()
    out = bn(x)
    mean = x.mean(dim=(0, 2, 3), keepdim=True)
    var = x.var(dim=(0, 2, 3), keepdim=True, unbiased=False)
    manual = (x - mean) / torch.sqrt(var + eps)
    return (bool(torch.allclose(out, manual, atol=1e-5)), out)

x = torch.tensor([[[[1., 2.], [3., 4.]]], [[[5., 6.], [7., 8.]]]])
print(solve(x))


## Why this works

In train mode BatchNorm2d normalizes with the BATCH statistics — per-channel mean and BIASED variance — plus eps inside the sqrt. Recomputing that by hand and getting `torch.allclose` to agree proves the convention; use `unbiased=True` and the match fails.
